# Topic embeddings in 2D with t-SNE

TO-DO:
- Try different random seeds
- Try differetn perplexity values

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize

DATA_DIR = Path("my_data/sgns")
PLOT_DIR = Path("plots")
RANDOM_STATE = 42
PERPLEXITY = 30 # effective number of neighbors
MAX_ITER = 1_000
TOPIC_COLUMNS = ["Topic_1", "Topic_2", "Topic_3"]

## 1. Load the topic vectors

In [2]:
topics_df = pd.read_csv(
    DATA_DIR / "topic_embeddings_normalized.csv", keep_default_na=False,
    dtype={"Topic_Id": "string", **{column: "string" for column in TOPIC_COLUMNS}},
)
embedding_columns = sorted(column for column in topics_df if column.startswith("embedding_"))
normalized_embeddings = topics_df[embedding_columns].to_numpy(dtype=np.float32)

print(f"Loaded {len(topics_df):,} topics with {normalized_embeddings.shape[1]} dimensions")

Loaded 1,000 topics with 150 dimensions


## 2. Run t-SNE

In [3]:
tsne = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    learning_rate="auto",
    init="pca",
    metric="euclidean",
    max_iter=MAX_ITER,
    random_state=RANDOM_STATE,
)
coordinates = tsne.fit_transform(normalized_embeddings)
projection_df = topics_df[["Topic_Id", *TOPIC_COLUMNS]].copy()
projection_df["Topic_path"] = projection_df[TOPIC_COLUMNS].apply(
    lambda row: " / ".join(value for value in row if value), axis=1
)
projection_df[["TSNE_1", "TSNE_2"]] = coordinates
print(f"Final KL divergence: {tsne.kl_divergence_:.4f}")

Final KL divergence: 1.2226


## 3. Plot and save

Color represents `Topic_1`; marker shape distinguishes `Topic_2` labels within each main topic. Shapes are assigned alphabetically within each `Topic_1` and reused across main topics. The legend shows only main topics; clicking one toggles all its subtopics; hover shows the full topic path.

Output:

- `plots/topic_embeddings_tsne.html`: interactive plot that can be opened in a browser.


In [ ]:
# Reuse the shape palette across main topics, with a distinct shape for
# every Topic_2 label within a given Topic_1. Sort labels for stable assignment.
marker_symbols = [
    "circle", "square", "diamond", "cross", "x",
    "triangle-up", "triangle-down", "triangle-left", "triangle-right",
    "pentagon", "hexagon", "star", "hexagram", "star-triangle-up",
    "star-triangle-down", "star-square", "star-diamond", "diamond-tall",
    "diamond-wide", "hourglass", "bowtie", "circle-cross", "circle-x",
    "square-cross", "square-x", "diamond-cross", "diamond-x",
]
topic2_symbols = {}
for topic1, group in projection_df.groupby("Topic_1", sort=False):
    topic2_labels = sorted(group["Topic_2"].unique())
    if len(topic2_labels) > len(marker_symbols):
        raise ValueError(f"Add more marker symbols for {topic1}: {len(topic2_labels)} Topic_2 labels")
    topic2_symbols[topic1] = dict(zip(topic2_labels, marker_symbols))

fig = px.scatter(
    projection_df,
    x="TSNE_1", y="TSNE_2", color="Topic_1", symbol="Topic_2",
    custom_data=["Topic_1", "Topic_2"],
    render_mode="svg",
    hover_name="Topic_path",
    hover_data={"Topic_Id": False, "Topic_1": False, "Topic_2": False, "TSNE_1": False, "TSNE_2": False},
    title=f"Topic embeddings · t-SNE (perplexity={PERPLEXITY}, seed={RANDOM_STATE})",
    labels={"Topic_1": "Main topic", "Topic_2": "Subtopic", "TSNE_1": "t-SNE 1", "TSNE_2": "t-SNE 2"},
    template="plotly_white",
    height=800,
)
# Each trace contains one (Topic_1, Topic_2) pair.
# Show one legend entry per main topic; clicking it toggles all its subtopics.
seen_topic1 = set()
for trace in fig.data:
    topic1, topic2 = trace.customdata[0][:2]
    trace.marker.symbol = topic2_symbols[topic1][topic2]
    trace.name = topic1
    trace.legendgroup = topic1
    trace.showlegend = topic1 not in seen_topic1
    seen_topic1.add(topic1)
fig.update_traces(marker=dict(size=7, opacity=0.8))
fig.update_layout(legend_title_text="Main topic", legend_groupclick="togglegroup")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
# projection_df.to_csv(DATA_DIR / "topic_embeddings_tsne.csv", index=False)
fig.write_html(PLOT_DIR / "topic_embeddings_tsne.html", include_plotlyjs=True)
fig.show()

<plotly.graph_objs._figure.Figure>